#### Plotting simple profiles of the ocean currents at a given dFAD time. 
1. Simple profile plots
    - Plot of the entire domain currents
    - plot of vertical profile of speed 
2. Making Annimation of dFAD traveling over current


In [1]:
import numpy as np 
import pandas as pd 
import xarray as xr 
import matplotlib.pyplot as plt 
import geopandas as gpd
from functions.funcs import *
import matplotlib.animation as animation
import imageio_ffmpeg
from matplotlib import rcParams
rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
%matplotlib notebook

In [2]:
import imageio_ffmpeg
from matplotlib import rcParams

rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

In [11]:
ds = xr.open_dataset(r"Data\cmems(3).nc")
data = gpd.read_parquet(r"Data\Mapped_4hr_period.parquet")
data2 = gpd.read_parquet(r"Data\Mapped_4hr_period2.parquet")

In [ ]:
data = Add_x_y_speed_collums(data)
data2 = Add_x_y_speed_collums(data2)

Index(['level_0', 'index', 'Shape_Leng', 'Set_3', 'BuoyName', 'Name_ID',
       'MinOfTimes', 'MaxOfTimes', 'MinOfDate', 'MaxOfDate', 'Yr_min',
       'Mon_min', 'Day_min', 'Yr_max', 'Mon_max', 'Day_max', 'Diff_days',
       'Distance_n', 'geometry', 'numpoints', 'SampleFreq', 'x_deg', 'y_deg',
       'x_km', 'y_km', 'xy_km', 'timelist', 'mapped_v', 'mapped_u', 'x_speed',
       'y_speed', 'xy_speed'],
      dtype='object')

In [4]:
vo  = ds['vo']
uo = ds['uo']

In [5]:
## plotting functions - First grad a test time 
data = add_time_collumns(data)
test_time = data.at[1,"timelist"][0]
print(type(test_time))

<class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [88]:
def Plotting_xy_currents(uo: xr.Dataset, vo : xr.Dataset, time: pd.Timestamp, depth: int, ax : plt.Axes):
    """Plot to create a profile of whole domains currents at specified depth"""
    ufield = uo.sel( depth = depth, time = time,method = "nearest")
    vfield = vo.sel( depth = depth, time = time,method = "nearest")
    latitude = uo["latitude"].to_numpy()
    longitude = uo["longitude"].to_numpy()
    latlocs, lonlocs = np.meshgrid( longitude, latitude)
    plot = ax.quiver(latlocs,lonlocs ,ufield,  vfield,headaxislength = 3, headlength = 3)
    ax.set_title("Model Currents")
    return plot
    


In [98]:
def Plotting_vertical_profile(uo: xr.Dataset, vo : xr.Dataset, lat : float, lon :float , time: pd.Timestamp,  ax : plt.Axes): 
    """Plots a vertial porfile of """
    ufield = uo.sel( longitude = lon, latitude = lat, time = time,method = "nearest")
    vfield = vo.sel( longitude = lon, latitude = lat, time = time,method = "nearest")
    yprofile = ax.plot(vfield,-vfield["depth"], label = "y speed", color = "g")
    xprofile = ax.plot(ufield,-ufield["depth"], label = "x speed", color = "b")
    ax.axvline(x = 0, alpha = 0.18, color = "k") #ymax = -max(vfield["depth"]), ymin = min(vfield["depth"]),
    ax.grid(alpha = 0.1)
    ax.set_ylabel("Depth")
    ax.set_xlabel("Speed")
    #ax.legend(loc = 'upper right')
    return xprofile, yprofile


In [8]:
def Add_lat_lon_collumns(data = pd.DataFrame):
    lats = []
    lons = []
    for i in range(len(data)):
        lat = []
        lon = []
        line = data.at[i,"geometry"]
        x,y = line.xy
        lats.append(y)
        lons.append(x)
    data["lat"] = lats
    data["lon"] = lons
    return data

In [89]:
def OneTrajectory(ax, ds,index, window:int = None , itime:int =None):
    line = ds.at[index,'geometry']
    x,y = line.xy
    if window !=None: ## adds padding to end of the array for sliding window
        x= np.array(x)
        y=np.array(y)
        nans = np.full(window,np.nan)
        x = np.concatenate((x,nans))
        nans = np.full(window,np.nan)
        y = np.concatenate((y,nans))
        x= x[itime:itime+window]
        y = y[itime:itime+window]
    dfadline = ax.plot(x,y)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("latitude")
    return dfadline

In [9]:
data = Add_lat_lon_collumns(data)
test_lat = data.at[1,"lat"][0]
test_lon = data.at[1,"lon"][0]
fig, ax = plt.subplots()
ax = Plotting_vertical_profile(uo,vo,test_lat,test_lon,time = test_time, ax= ax)

<IPython.core.display.Javascript object>

In [23]:
fig, ax = plt.subplots()
plot= Plotting_xy_currents(uo, vo, test_time, 15, ax)
ax = OneTrajectory(data, 1, ax, window = 5, itime= 0)

<IPython.core.display.Javascript object>

#### want to make an animation with that changes through time of the currents 

In [86]:
## first get a the time I want to animate  
dfadidex = 20
times = data.at[dfadidex,"timelist"]
artists = []
fig, ax = plt.subplots()

for n,time in enumerate(times): ## giving n doesnt quite work because if times is not the start time. 
    current = Plotting_xy_currents(uo, vo, time, 15, ax)
    dfad = OneTrajectory(ax, data,dfadidex,window = 12,itime=n,)
    ann = ax.annotate(f"{times[n]:%Y :%m :%d :%H}",(0.15, 1.03), xycoords="axes fraction", ha="center")
    artists.append([current,dfad[0],ann])


print(artists)

ani = animation.ArtistAnimation(fig =fig, artists = artists, blit= False)
ani.save(rf"..\Figures\Animations\Animation_{dfadidex}.mp4", writer = "ffmpeg")

<IPython.core.display.Javascript object>

C:\Users\czerfass\AppData\Local\Temp\2\ipykernel_6012\1034392704.py:9: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()


[[<matplotlib.quiver.Quiver object at 0x000001ACECF0F750>, <matplotlib.lines.Line2D object at 0x000001ACECF0F890>, Text(0.15, 1.03, '02')], [<matplotlib.quiver.Quiver object at 0x000001ACEEC302D0>, <matplotlib.lines.Line2D object at 0x000001ACEEC30410>, Text(0.15, 1.03, '06')], [<matplotlib.quiver.Quiver object at 0x000001ACEEC30E10>, <matplotlib.lines.Line2D object at 0x000001ACEEC30F50>, Text(0.15, 1.03, '10')], [<matplotlib.quiver.Quiver object at 0x000001ACEEC31950>, <matplotlib.lines.Line2D object at 0x000001ACEEC31A90>, Text(0.15, 1.03, '14')], [<matplotlib.quiver.Quiver object at 0x000001ACEEC32490>, <matplotlib.lines.Line2D object at 0x000001ACEEC325D0>, Text(0.15, 1.03, '18')], [<matplotlib.quiver.Quiver object at 0x000001ACEEC32FD0>, <matplotlib.lines.Line2D object at 0x000001ACEEC33110>, Text(0.15, 1.03, '22')], [<matplotlib.quiver.Quiver object at 0x000001ACEEC33B10>, <matplotlib.lines.Line2D object at 0x000001ACECF0F9D0>, Text(0.15, 1.03, '02')], [<matplotlib.quiver.Quiver

C:\Users\czerfass\AppData\Local\Temp\2\ipykernel_6012\2605784613.py:17: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  ani.save(rf"..\Figures\Animations\Animation_{dfadidex}.mp4", writer = "ffmpeg")


#### adding profile plot to the side also 

In [91]:
from matplotlib.gridspec import GridSpec

In [102]:
dfadidex = 40
times = data.at[dfadidex,"timelist"]
lats = data.at[dfadidex, "lat"]
lons = data.at[dfadidex, "lon"]
artists = []

fig = plt.figure()
gs = GridSpec(3,3)
ax1 = fig.add_subplot(gs[:,:-1])
ax2 = fig.add_subplot(gs[:,-1])

for n,time in enumerate(times): ## giving n doesnt quite work because if times is not the start time. 
    current = Plotting_xy_currents(uo, vo, time, 15, ax1)
    dfad = OneTrajectory(ax1, data,dfadidex,window = 12,itime=n,)
    ann = ax1.annotate(f"{times[n]:%Y :%m :%d :%H}",(0.10, 1.03), xycoords="axes fraction", ha="center")
    xprofile, yprofile = Plotting_vertical_profile(uo,vo,lat = lats[n], lon= lons[n], time = times[n], ax=  ax2)
    artists.append([current,dfad[0],ann, xprofile[0], yprofile[0]])


print(artists)

ani = animation.ArtistAnimation(fig =fig, artists = artists, blit= False)
ani.save(rf"..\Figures\Animations\Depths_{dfadidex}.mp4", writer = "ffmpeg")

<IPython.core.display.Javascript object>

[[<matplotlib.quiver.Quiver object at 0x000001ACF3023250>, <matplotlib.lines.Line2D object at 0x000001ACF2B43610>, Text(0.15, 1.03, '2023 :07 :12 :13'), <matplotlib.lines.Line2D object at 0x000001ACF2B43ED0>, <matplotlib.lines.Line2D object at 0x000001ACF2B43D90>], [<matplotlib.quiver.Quiver object at 0x000001ACF2B43890>, <matplotlib.lines.Line2D object at 0x000001ACF2B43C50>, Text(0.15, 1.03, '2023 :07 :12 :17'), <matplotlib.lines.Line2D object at 0x000001ACF2B41450>, <matplotlib.lines.Line2D object at 0x000001ACF2B43390>], [<matplotlib.quiver.Quiver object at 0x000001ACF2B41A90>, <matplotlib.lines.Line2D object at 0x000001ACF2B416D0>, Text(0.15, 1.03, '2023 :07 :12 :20'), <matplotlib.lines.Line2D object at 0x000001ACF2B41F90>, <matplotlib.lines.Line2D object at 0x000001ACF2B41E50>], [<matplotlib.quiver.Quiver object at 0x000001ACF2B41D10>, <matplotlib.lines.Line2D object at 0x000001ACF2B42350>, Text(0.15, 1.03, '2023 :07 :13 :00'), <matplotlib.lines.Line2D object at 0x000001ACF2B4221

In [10]:
first = data["x_km"][40]
first

array([-0.91068645, -2.52746068, -3.75616462, -4.45558071, -2.41848965,
       -4.90703211, -5.33735648, -2.41292991, -4.19760848, -4.0964211 ,
       -2.71982791, -4.10198084, -4.27544493, -2.90329953, -4.04415948,
       -4.08641355, -4.91259186, -6.07569079, -3.86513565, -3.98077837,
       -4.68909006, -3.82176963, -4.36551282, -4.04193558, -0.613796  ,
       -3.85846395, -3.638298  , -2.54413992, -6.00119019, -3.65275334,
       -0.18903138, -4.85921829, -4.97708492, -3.39144526, -4.41666249,
       -4.66129132, -2.36289219, -2.40181042, -4.08196576, -3.80509039,
       -3.32472831, -4.05527897, -4.5545442 , -5.23060935, -3.59493198,
       -4.8503227 , -3.88737464, -3.89960608, -4.1664739 , -4.75135922,
       -3.87847904, -1.88364206, -2.39402677, -4.6201492 , -2.87216496,
       -3.51820748, -4.93816669, -3.09344286, -3.29136983, -4.36773672,
       -4.11310034, -4.91036796, -5.13386976, -3.93963625, -3.38588552,
       -4.51117817, -5.50748472, -4.48115554, -4.01636075, -3.31